<a href="https://colab.research.google.com/github/23MH1A05M8/-Product-Review-Event-Processor-with-Sentiment-Analysis-and-Message-Queues/blob/main/Day5_HF_Pulls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers requests sentence-transformers

import os
import getpass

if 'HF_TOKEN' not in os.environ:
    os.environ['HF_TOKEN'] = getpass.getpass('HF token (read scope): ')

HF token (read scope): ··········


In [ ]:
# Note: API endpoint could not be resolved in this Colab runtime.
# Local inference was used for completion of the lab.
import os
import requests
import time

HF_TOKEN = os.environ['HF_TOKEN']

def hf_zero_shot_api(text, labels):
    r = requests.post(
        'https://api-inference.huggingface.co/models/facebook/bart-large-mnli',
        headers={'Authorization': f'Bearer {HF_TOKEN}'},
        json={
            'inputs': text,
            'parameters': {
                'candidate_labels': labels
            }
        }
    )
    return r.json()

# Sample resumes
resumes = [
    'Built React dashboards for 3 startups',
    'Implemented Spring Boot microservices in Java for fintech app',
    'Trained CNN for image classification using PyTorch, 87% accuracy',
    'Cleaned 100k row dataset using pandas + plotted in seaborn for monthly reports',
    'Wrote SQL queries against PostgreSQL, optimised 3 slow queries by 10x'
]

labels = [
    'frontend dev',
    'backend dev',
    'data analyst',
    'ML engineer'
]

start = time.time()

for r in resumes:
    result = hf_zero_shot_api(r, labels)

    if 'error' in result:
        print("Error:", result['error'])
    else:
        print(
            f'{r[:50]:50} -> '
            f'{result["labels"][0]} '
            f'({result["scores"][0]:.2f})'
        )

print(f'\nAPI time: {time.time()-start:.2f}s')

In [11]:
# Local Model Loading
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Model loaded successfully!


In [7]:
#Resume Classification
import time

resumes = [
    'Built React dashboards for 3 startups',
    'Implemented Spring Boot microservices in Java for fintech app',
    'Trained CNN for image classification using PyTorch, 87% accuracy',
    'Cleaned 100k row dataset using pandas + plotted in seaborn for monthly reports',
    'Wrote SQL queries against PostgreSQL, optimised 3 slow queries by 10x'
]

labels = [
    'frontend dev',
    'backend dev',
    'data analyst',
    'ML engineer'
]

start = time.time()

for r in resumes:
    result = classifier(r, candidate_labels=labels)

    print(
        f'{r[:50]:50} -> '
        f'{result["labels"][0]} '
        f'({result["scores"][0]:.2f})'
    )

print(f'\nLocal time: {time.time()-start:.2f}s')

Built React dashboards for 3 startups              -> frontend dev (0.88)
Implemented Spring Boot microservices in Java for  -> backend dev (0.63)
Trained CNN for image classification using PyTorch -> ML engineer (0.40)
Cleaned 100k row dataset using pandas + plotted in -> data analyst (0.56)
Wrote SQL queries against PostgreSQL, optimised 3  -> data analyst (0.45)

Local time: 13.83s


In [8]:
# Sentiment Analysis
from transformers import pipeline

sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

answers = [
    'I really enjoyed working on the team and shipped 3 features.',
    'I was the only one writing code; everyone else was slow.',
    'I learned a lot from my mentor and grew technically.',
    "I had to redo most of my teammate's work because it was wrong.",
    'My internship was great — would recommend it to anyone.'
]

print("Sentiment scores:\n")

for a in answers:
    result = sentiment(a)[0]

    print(
        f'[{result["label"]} {result["score"]:.2f}] '
        f'{a}'
    )

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Sentiment scores:

[POSITIVE 1.00] I really enjoyed working on the team and shipped 3 features.
[NEGATIVE 1.00] I was the only one writing code; everyone else was slow.
[POSITIVE 1.00] I learned a lot from my mentor and grew technically.
[NEGATIVE 1.00] I had to redo most of my teammate's work because it was wrong.
[POSITIVE 1.00] My internship was great — would recommend it to anyone.


In [9]:
# Timing Comparison
import time

def time_call(fn, n_runs=3):
    times = []

    for _ in range(n_runs):
        start = time.time()
        fn()
        times.append(time.time() - start)

    return min(times), sum(times)/len(times)

def call_local():
    classifier(
        "Built React dashboards",
        candidate_labels=["frontend dev", "backend dev"]
    )

local_min, local_avg = time_call(call_local)

print("Inference Timing")
print("-" * 30)
print(f"Local min: {local_min:.2f}s")
print(f"Local avg: {local_avg:.2f}s")

Inference Timing
------------------------------
Local min: 0.85s
Local avg: 0.87s
